# GPT-2 Activation Extraction and Analysis


In [ ]:
import os
import torch
from transformers import GPT2Model, GPT2Tokenizer

modelName = "gpt2"
# If running in Google Colab, you can store your HF_TOKEN in Colab Secrets and authorize it
huggingFaceToken = os.environ.get("HF_TOKEN", None)

tokenizer = GPT2Tokenizer.from_pretrained(modelName, token=huggingFaceToken)
languageModel = GPT2Model.from_pretrained(modelName, torch_dtype=torch.float16, token=huggingFaceToken)
languageModel.eval() # Put model in evaluation mode

print("Model loaded successfully in FP16.")

In [ ]:
inputText = "Parallel computing optimizes systems."
tokenizedInput = tokenizer(inputText, return_tensors="pt")

# Map token IDs back to human-readable strings to inspect tokenization
tokensList = [tokenizer.decode([tokenId]) for tokenId in tokenizedInput["input_ids"][0]]

print(f"Input Text: '{inputText}'")
print(f"Tokens:     {tokensList}")
print(f"Input IDs:  {tokenizedInput['input_ids'].tolist()[0]}")

In [ ]:
with torch.no_grad():
    modelOutputs = languageModel(**tokenizedInput, output_hidden_states=True)

# hidden_states contains a tuple of activation tensors for all layers
activationTensors = modelOutputs.hidden_states

print(f"Total hidden states extracted: {len(activationTensors)}")
print("Note: Index 0 represents embedding outputs; indexes 1-12 represent block outputs.")

In [ ]:
embeddingOutput = activationTensors[0]
firstBlockOutput = activationTensors[1]
finalBlockOutput = activationTensors[-1]

print(f"Embedding Output Shape (Layer 0): {list(embeddingOutput.shape)}")
print(f"First Block Output Shape (Layer 1): {list(firstBlockOutput.shape)}")
print(f"Final Block Output Shape (Layer 12): {list(finalBlockOutput.shape)}")

In [ ]:
print(f"{'Layer':<8} | {'Min':<10} | {'Max':<10} | {'Mean':<10} | {'Std Dev':<10}")
print("-" * 58)

for layerIdx, activations in enumerate(activationTensors):
    # Convert to float32 for stable calculation
    floatActivations = activations.to(torch.float32)
    minValue = floatActivations.min().item()
    maxValue = floatActivations.max().item()
    meanValue = floatActivations.mean().item()
    stdValue = floatActivations.std().item()
    
    layerName = f"L{layerIdx}" if layerIdx > 0 else "Embed"
    print(f"{layerName:<8} | {minValue:<10.4f} | {maxValue:<10.4f} | {meanValue:<10.4f} | {stdValue:<10.4f}")

In [ ]:
specificLayerActivations = activationTensors[1]
firstTokenSample = specificLayerActivations[0, 0, :10]

print("Raw FP16 Activation Matrix (First 10 values of the first token):")
print(firstTokenSample.tolist())